# Filtering by checkv quality and size

In [5]:
import pandas as pd
import re

samples = [
    'AP1','AP2','AP3','AP4','AP5',
    'BP1','BP2','BP3','BP4','BP5',
    'MP1','MP2','MP3','MP4','MP5',
    'PP1','PP2','PP3','PP4','PP5'
]

def change_name(string,sample):
    string = f'{sample}_{string}'
    return string

def parse_checkv_results(samplelist):
    df = pd.DataFrame({})
    temp_df = pd.DataFrame({})

    for sample in samplelist:
        temp_df = pd.read_csv(f'/MP_Data/Dados_David/viral_analysis/results/{sample}/final_filtered/final_quality_summary.tsv',sep='\t')
        temp_df['length'] = temp_df['contig_id'].str.split('_').str[3]
        temp_df['length'] = temp_df['length'].astype(int)
        temp_df['contig_id'] = temp_df['contig_id'].apply(lambda x: change_name(x, sample))

        if df.empty:
            df = temp_df
        else:
            df = pd.concat([df, temp_df], ignore_index=True)

    return df

quality_df = parse_checkv_results(samples)

#Maintain scaffold if >= 10kb or if qual = medium, high, complete
quality_df = quality_df[(quality_df['contig_length'] >= 10000) | (quality_df['checkv_quality'].isin(['High-quality','Complete','Medium-quality']))]

#Maintain only the Longest sequence from the cluster calculated by anicalc.py
miuvigs = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/clustering/miuvig_list.tsv', sep='\t')
quality_df = quality_df[quality_df['contig_id'].isin(miuvigs['miuvig_id'])]

quality_df

,contig_id,contig_length,provirus,proviral_length,gene_count,viral_genes,host_genes,checkv_quality,miuvig_quality,completeness,completeness_method,contamination,kmer_freq,warnings,length
1,AP1_NODE_70_length_138287_cov_3.937811,29852,Yes,10182.0,24,7,14,Low-quality,Genome-fragment,15.71,HMM-based (lower-bound),65.89,1.00,NaN,138287
2,AP1_NODE_159_length_92405_cov_3.227497,56568,Yes,35632.0,47,6,10,Medium-quality,Genome-fragment,59.20,AAI-based (medium-confidence),37.01,1.00,NaN,92405
3,AP1_NODE_223_length_71899_cov_28.628643,61318,Yes,26036.0,68,4,12,Low-quality,Genome-fragment,31.59,HMM-based (lower-bound),57.54,1.01,NaN,71899
4,AP1_NODE_341_length_56568_cov_4.154303,33470,Yes,22803.0,39,15,7,Medium-quality,Genome-fragment,50.21,AAI-based (medium-confidence),31.87,1.06,NaN,56568
6,AP1_NODE_4720_length_11954_cov_1.134776,11194,No,NaN,3,0,0,Not-determined,Genome-fragment,NaN,NaN,0.00,1.00,no viral genes detected,11954
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39033,PP5_NODE_2416_length_10065_cov_0.977863,10065,No,NaN,10,6,0,Low-quality,Genome-fragment,10.90,AAI-based (medium-confidence),0.00,1.00,NaN,10065
39034,PP5_NODE_2445_length_10018_cov_0.913153,10018,No,NaN,11,1,0,Low-quality,Genome-fragment,4.21,AAI-based (medium-confidence),0.00,1.00,NaN,10018
39440,PP5_NODE_822_length_16605_cov_1.932273,16605,No,NaN,22,7,0,Low-quality,Genome-fragment,6.57,HMM-based (lower-bound),0.00,1.00,NaN,16605
39441,PP5_NODE_2289_length_10333_cov_1.175779,10333,No,NaN,12,0,0,Not-determined,Genome-fragment,NaN,NaN,0.00,1.00,no viral genes detected,10333


## Add taxonomic information by genomad

In [6]:

def parse_genomad_results(samplelist):
    df = pd.DataFrame({})
    temp_df = pd.DataFrame({})

    for sample in samplelist:
        temp_df = pd.read_csv(f'/MP_Data/Dados_David/viral_analysis/results/{sample}/genomad/filtered_checkv_summary/filtered_checkv_virus_summary.tsv',sep='\t')
        temp_df['length'] = temp_df['seq_name'].str.split('_').str[3]
        temp_df['length'] = temp_df['length'].astype(int)
        temp_df['seq_name'] = temp_df['seq_name'].apply(lambda x: change_name(x, sample))

        if df.empty:
            df = temp_df
        else:
            df = pd.concat([df, temp_df], ignore_index=True)

    return df
tax_df = parse_genomad_results(samples)

tax_df

,seq_name,length,topology,coordinates,n_genes,genetic_code,virus_score,fdr,n_hallmarks,marker_enrichment,taxonomy
0,AP1_NODE_4947_length_11656_cov_2.939804,11656,No terminal repeats,NaN,21,11,0.9837,NaN,4,35.0137,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
1,AP1_NODE_5071_length_11509_cov_3.290546,11509,No terminal repeats,NaN,13,11,0.9837,NaN,2,22.3187,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
2,AP1_NODE_13833_length_6576_cov_6.247480,6576,No terminal repeats,NaN,10,11,0.9837,NaN,6,17.0536,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
3,AP1_NODE_34962_length_3914_cov_2.592025,3914,No terminal repeats,NaN,7,15,0.9836,NaN,1,12.0280,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
4,AP1_NODE_13914_length_6553_cov_1.373171,6553,No terminal repeats,NaN,12,11,0.9836,NaN,1,20.6143,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
...,...,...,...,...,...,...,...,...,...,...,...
50668,PP5_NODE_18341_length_3884_cov_0.851211,3884,No terminal repeats,NaN,5,11,0.7183,NaN,0,0.0000,Unclassified
50669,PP5_NODE_19757_length_3756_cov_1.518325,3756,No terminal repeats,NaN,3,11,0.7147,NaN,0,0.0000,Unclassified
50670,PP5_NODE_29789_length_3099_cov_0.751009,3099,No terminal repeats,NaN,2,11,0.7085,NaN,0,0.0000,Unclassified
50671,PP5_NODE_1703_length_11827_cov_1.210000,11827,No terminal repeats,NaN,1,11,0.7063,NaN,0,1.6512,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...


In [7]:
qual_plus_tax = pd.merge(quality_df, tax_df, left_on='contig_id',right_on='seq_name',how='left')
#qual_plus_tax = qual_plus_tax[qual_plus_tax['taxonomy'].notna()]
qual_plus_tax['taxonomy'].value_counts()

qual_plus_tax.shape

(7093, 26)

In [8]:
#Remove non bacteriophage taxonomies (Possible false positives)
non_phages = [
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Algavirales;Phycodnaviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;;',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Imitervirales;Mimiviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Imitervirales;',
    'Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;Priklausovirales;',
    'Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Imitervirales;Allomimiviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Pimascovirales;Iridoviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Pokkesviricetes;Asfuvirales;Asfarviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Polintoviricetes;Orthopolintovirales;Adintoviridae',
    'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;Megaviricetes;Pimascovirales;Marseilleviridae',
    'Viruses;Riboviria;Pararnavirae;Artverviricota;Revtraviricetes;Ortervirales;Retroviridae',
    'Viruses;Riboviria;Orthornavirae;Kitrinoviricota;Tolucaviricetes;Tolivirales;Tombusviridae'
    
]

qual_plus_tax = qual_plus_tax[~qual_plus_tax['taxonomy'].isin(non_phages)]
qual_plus_tax = qual_plus_tax[~qual_plus_tax['taxonomy'].isna()]

qual_plus_tax.shape

(6650, 26)

# Check votus length range

In [9]:
qual_plus_tax['contig_length'].describe()

count      6650.000000
mean      25147.043308
std       26463.368550
min        4537.000000
25%       12375.500000
50%       16805.500000
75%       28825.250000
max      574697.000000
Name: contig_length, dtype: float64

# Check taxonomy

In [10]:
qual_plus_tax['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  6034
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       410
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae                   56
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae                 23
Unclassified                                                                                         18
Viruses;;;;;;                                                                                        12
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae                      12
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;                        11
Viruses;Varidnaviria;Bamfordvirae;;;;                                                                 9
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;;; 

# NCLDVs and Virophages will be identified based on phylogeny!!

# Checking if all viral contigs have more the 1X depth in each sample

In [11]:
viral_set = qual_plus_tax['contig_id'].tolist()
viral_set = set(viral_set)

#Load viral TMM normalized dataset
viral_map_df = pd.read_csv(
    "/MP_Data/Dados_David/viral_analysis/post_processing/viral_map_data/sectet_viral_TMM_norm.tsv",
    sep="\t", index_col=0
)
viral_map_df  = viral_map_df[viral_map_df.index.isin(viral_set)]#.reset_index(drop=True)
viral_map_df

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
AP1_NODE_1015_length_28855_cov_2.442704,23.034463,19.933241,11.544837,24.242328,13.499963,20.308196,1.436280,0.862032,1.302371,1.457831,...,0.164928,0.195510,0.127637,0.177858,0.000000,0.112592,0.024786,0.045322,0.000000,0.000000
AP1_NODE_1019_length_28805_cov_4.940442,27.828627,13.375569,6.581884,48.419226,25.133198,47.414233,14.247897,18.102673,11.028587,6.334023,...,0.000000,0.027930,0.000000,0.000000,0.000000,0.000000,0.000000,0.045322,0.050264,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,13.146498,11.189679,6.979982,9.520267,6.303590,9.272369,1.407554,2.068877,1.662601,1.910261,...,0.000000,0.000000,0.063819,0.000000,0.000000,0.000000,0.000000,0.045322,0.025132,0.000000
AP1_NODE_1087_length_27770_cov_1.242123,7.528337,9.107878,9.501268,7.197452,7.899778,12.116655,0.459610,0.402282,0.166260,0.829455,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.022661,0.000000,0.045098
AP1_NODE_10_length_320914_cov_6.491204,538.894067,335.716380,426.787395,988.242938,594.241878,1216.870493,23.526265,25.113866,23.802905,35.566041,...,0.109952,0.111720,0.031909,0.000000,0.189987,0.056296,0.099143,0.158627,0.226187,0.157842
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PP5_NODE_924_length_15762_cov_1.839591,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.229805,0.258610,1.330081,0.201080,...,0.000000,0.111720,0.127637,0.148215,0.221652,0.112592,0.049572,14.661663,11.761727,9.966607
PP5_NODE_94_length_56131_cov_2.043533,0.000000,0.026023,0.053080,0.000000,0.000000,0.000000,0.229805,0.431016,0.304810,0.150810,...,23.089979,21.589895,30.441465,19.594005,34.324339,13.370324,26.520851,13.347326,9.826571,8.591125
PP5_NODE_953_length_15547_cov_2.636252,0.000000,0.026023,0.053080,0.065431,0.054108,0.000000,0.057451,0.114938,0.055420,0.000000,...,9.703289,11.563022,16.241830,9.278250,12.412492,6.642940,8.922903,5.076063,3.367674,2.750964
PP5_NODE_954_length_15543_cov_1.573560,0.074909,0.000000,0.000000,0.130863,0.054108,0.085329,0.430884,0.143672,0.083130,0.351890,...,2.748807,1.703730,2.169832,1.245005,1.551561,0.731849,1.511936,11.149209,9.625516,7.193094


In [12]:
#Load viral depth dataset
viral_depth_df = pd.read_csv(
    "/MP_Data/Dados_David/viral_analysis/mapped_reads/sectet_contigs_coverage.txt",
    sep="\t", index_col=0
)

viral_depth_df  = viral_depth_df[viral_depth_df.index.isin(viral_set)]#.reset_index(drop=True)
viral_depth_df


,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
contig_id,,,,,,,,,,,,,,,,,,,,,
AP1_NODE_1015_length_28855_cov_2.442704,3.104372,3.845149,2.163526,3.711966,2.519073,3.583801,0.233478,0.154503,0.236474,0.289462,...,0.025257,0.021634,0.020658,0.024769,0.000000,0.015084,0.001812,0.006689,0.000000,0.000000
AP1_NODE_1019_length_28805_cov_4.940442,3.819543,2.629524,1.282359,7.536835,4.746606,8.516733,2.550235,3.249555,2.035526,1.290106,...,0.000000,0.004362,0.000000,0.000000,0.000000,0.000000,0.000000,0.010051,0.003839,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,1.821398,2.214616,1.370866,1.493484,1.192125,1.656740,0.246462,0.363789,0.300308,0.401030,...,0.000000,0.000000,0.010510,0.000000,0.000000,0.000000,0.000000,0.004064,0.001296,0.000000
AP1_NODE_1087_length_27770_cov_1.242123,1.066256,1.830485,1.854888,1.138921,1.543737,2.204164,0.082332,0.076213,0.032766,0.161151,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002462,0.000000,0.002462
AP1_NODE_10_length_320914_cov_6.491204,6.567083,5.891210,7.343187,13.592333,10.013150,19.520184,0.373206,0.399580,0.389691,0.644527,...,0.000608,0.000571,0.000302,0.000000,0.002825,0.000483,0.000664,0.001138,0.002407,0.001515
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PP5_NODE_924_length_15762_cov_1.839591,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.050474,0.075391,0.407315,0.056431,...,0.000000,0.036510,0.036254,0.041763,0.067384,0.038496,0.019344,5.880349,4.259352,3.994235
PP5_NODE_94_length_56131_cov_2.043533,0.000000,0.001358,0.005377,0.000000,0.000000,0.000000,0.018381,0.032386,0.026491,0.015184,...,2.174970,2.031386,2.479091,1.734696,2.807363,1.233240,2.773084,1.532824,1.020864,1.002215
PP5_NODE_953_length_15547_cov_2.636252,0.000000,0.005001,0.017796,0.019484,0.009028,0.000000,0.019549,0.020264,0.016042,0.000000,...,3.362343,3.968630,4.772228,2.913230,3.667468,2.223550,3.376177,2.143470,1.285705,1.136715


In [13]:
viral_depth = viral_depth_df
viral_norm_abund = viral_map_df

# ============================================================
# Identify origin sample
# ============================================================

viral_depth['origin_sample'] = (
    viral_depth.index
    .str.split('_')
    .str[0]
)

viral_map_df = viral_norm_abund.copy()
# ============================================================
# Apply cutoff only to non-origin samples
# ============================================================

for col in viral_map_df.columns:

    # AP1_A -> AP1
    # AP1_B -> AP1
    # AP2_A -> AP2
    col_origin = col.split('_')[0]

    # Contigs that DO NOT originate from this sample
    non_origin = viral_depth['origin_sample'] != col_origin

    # Apply coverage cutoff only to those contigs
    viral_map_df.loc[non_origin, col] = (
        viral_map_df.loc[non_origin, col]
        .where(
            viral_depth.loc[non_origin, col] >= 1,
            0
        )
    )
viral_map_df

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
AP1_NODE_1015_length_28855_cov_2.442704,23.034463,19.933241,11.544837,24.242328,13.499963,20.308196,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AP1_NODE_1019_length_28805_cov_4.940442,27.828627,13.375569,6.581884,48.419226,25.133198,47.414233,14.247897,18.102673,11.028587,6.334023,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,13.146498,11.189679,6.979982,9.520267,6.303590,9.272369,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AP1_NODE_1087_length_27770_cov_1.242123,7.528337,9.107878,9.501268,7.197452,7.899778,12.116655,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AP1_NODE_10_length_320914_cov_6.491204,538.894067,335.716380,426.787395,988.242938,594.241878,1216.870493,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PP5_NODE_924_length_15762_cov_1.839591,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.661663,11.761727,9.966607
PP5_NODE_94_length_56131_cov_2.043533,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,23.089979,21.589895,30.441465,19.594005,34.324339,13.370324,26.520851,13.347326,9.826571,8.591125
PP5_NODE_953_length_15547_cov_2.636252,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.703289,11.563022,16.241830,9.278250,12.412492,6.642940,8.922903,5.076063,3.367674,2.750964
PP5_NODE_954_length_15543_cov_1.573560,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.149209,9.625516,7.193094


# Define final viral list

In [14]:
viral_map_df = viral_map_df.reset_index()
viral_map_df = viral_map_df.rename(columns={'index':'contig_id'})
viral_map_df

,contig_id,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
0,AP1_NODE_1015_length_28855_cov_2.442704,23.034463,19.933241,11.544837,24.242328,13.499963,20.308196,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,AP1_NODE_1019_length_28805_cov_4.940442,27.828627,13.375569,6.581884,48.419226,25.133198,47.414233,14.247897,18.102673,11.028587,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,AP1_NODE_1026_length_28694_cov_1.324710,13.146498,11.189679,6.979982,9.520267,6.303590,9.272369,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,AP1_NODE_1087_length_27770_cov_1.242123,7.528337,9.107878,9.501268,7.197452,7.899778,12.116655,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,AP1_NODE_10_length_320914_cov_6.491204,538.894067,335.716380,426.787395,988.242938,594.241878,1216.870493,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6645,PP5_NODE_924_length_15762_cov_1.839591,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,14.661663,11.761727,9.966607
6646,PP5_NODE_94_length_56131_cov_2.043533,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,23.089979,21.589895,30.441465,19.594005,34.324339,13.370324,26.520851,13.347326,9.826571,8.591125
6647,PP5_NODE_953_length_15547_cov_2.636252,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,9.703289,11.563022,16.241830,9.278250,12.412492,6.642940,8.922903,5.076063,3.367674,2.750964
6648,PP5_NODE_954_length_15543_cov_1.573560,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.149209,9.625516,7.193094


- Check votus quantity by city

In [15]:
sample_dist = viral_map_df
#sample_dist = sample_dist.reset_index()
sample_dist['sample'] = sample_dist['contig_id'].str.split('_').str[0]
sample_dist = sample_dist['sample']
sample_dist.value_counts(ascending=False)

sample
MP1    1135
MP5    1049
AP5     740
MP2     621
MP4     551
AP2     409
AP3     348
MP3     323
AP4     245
AP1     228
BP2     154
BP1     147
PP2     122
PP1     120
PP5     118
BP4      99
PP3      90
BP3      67
PP4      60
BP5      24
Name: count, dtype: int64

In [16]:
final_votus = viral_map_df['contig_id'].tolist()
len(final_votus)

6650

- Joining relative abundance, quality and taxonomy tables

In [17]:
qual_plus_tax_plus_abund = pd.merge(qual_plus_tax, viral_map_df, left_on='seq_name', right_on='contig_id')
#qual_plus_tax_plus_abund.to_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/quality_plus_taxonomy.tsv",sep='\t',index=False)
qual_plus_tax_plus_abund.shape

(6650, 88)

- Extracting only the taxonomies

In [32]:
tax_df_final = tax_df[tax_df['seq_name'].isin(final_votus)]
#tax_df_final.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/final_votus_genomad_results.tsv', sep='\t',index=False)
tax_df_final

,seq_name,length,topology,coordinates,n_genes,genetic_code,virus_score,fdr,n_hallmarks,marker_enrichment,taxonomy
0,AP1_NODE_4947_length_11656_cov_2.939804,11656,No terminal repeats,NaN,21,11,0.9837,NaN,4,35.0137,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
1,AP1_NODE_5071_length_11509_cov_3.290546,11509,No terminal repeats,NaN,13,11,0.9837,NaN,2,22.3187,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
14,AP1_NODE_1422_length_23637_cov_2.675202,23637,No terminal repeats,NaN,31,11,0.9834,NaN,3,43.4383,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
18,AP1_NODE_1411_length_23714_cov_3.915080,23714,No terminal repeats,NaN,29,11,0.9833,NaN,5,39.6908,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
50,AP1_NODE_4021_length_13131_cov_2.998154,13131,No terminal repeats,NaN,19,11,0.9830,NaN,2,28.6021,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
...,...,...,...,...,...,...,...,...,...,...,...
50594,PP5_NODE_2063_length_10826_cov_2.202262,10826,DTR,NaN,23,11,0.8687,NaN,1,11.5669,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...
50603,PP5_NODE_1033_length_15046_cov_1.703264,15046,No terminal repeats,NaN,24,11,0.8538,NaN,0,1.0603,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
50612,PP5_NODE_2289_length_10333_cov_1.175779,10333,No terminal repeats,NaN,12,11,0.8479,NaN,0,3.1862,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
50623,PP5_NODE_2409_length_10076_cov_1.213489,10076,No terminal repeats,NaN,16,11,0.8329,NaN,0,4.8815,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...


- Extracting only the qualities

In [27]:
quality_df_final = quality_df[quality_df['contig_id'].isin(final_votus)]
#quality_df_final.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/quality/final_votus_checkv_results.tsv', sep='\t',index=False)
quality_df_final

,contig_id,contig_length,provirus,proviral_length,gene_count,viral_genes,host_genes,checkv_quality,miuvig_quality,completeness,completeness_method,contamination,kmer_freq,warnings,length
2,AP1_NODE_159_length_92405_cov_3.227497,56568,Yes,35632.0,47,6,10,Medium-quality,Genome-fragment,59.20,AAI-based (medium-confidence),37.01,1.00,NaN,92405
3,AP1_NODE_223_length_71899_cov_28.628643,61318,Yes,26036.0,68,4,12,Low-quality,Genome-fragment,31.59,HMM-based (lower-bound),57.54,1.01,NaN,71899
7,AP1_NODE_10_length_320914_cov_6.491204,320914,No,NaN,378,30,25,High-quality,High-quality,90.22,AAI-based (medium-confidence),0.00,1.01,NaN,320914
9,AP1_NODE_37_length_195771_cov_2.985223,195771,No,NaN,200,19,18,High-quality,High-quality,91.50,HMM-based (lower-bound),0.00,1.00,NaN,195771
10,AP1_NODE_74_length_133942_cov_3.123245,133942,No,NaN,114,9,11,High-quality,High-quality,96.81,AAI-based (high-confidence),0.00,1.00,NaN,133942
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39032,PP5_NODE_2409_length_10076_cov_1.213489,10076,No,NaN,16,1,1,Low-quality,Genome-fragment,21.68,AAI-based (high-confidence),0.00,1.00,NaN,10076
39033,PP5_NODE_2416_length_10065_cov_0.977863,10065,No,NaN,10,6,0,Low-quality,Genome-fragment,10.90,AAI-based (medium-confidence),0.00,1.00,NaN,10065
39034,PP5_NODE_2445_length_10018_cov_0.913153,10018,No,NaN,11,1,0,Low-quality,Genome-fragment,4.21,AAI-based (medium-confidence),0.00,1.00,NaN,10018
39440,PP5_NODE_822_length_16605_cov_1.932273,16605,No,NaN,22,7,0,Low-quality,Genome-fragment,6.57,HMM-based (lower-bound),0.00,1.00,NaN,16605


- Generating file with the final viral scaffolds

In [ ]:
with open('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/votus_list.txt','w') as file:
    for line in final_votus:
        file.write(f'{line}\n')

# Adding NCLDV and virophage vOTUs

In [30]:
ncldvs = pd.read_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/ncldvs_virophages/ncldv_scaffolds.tsv",sep='\t')
ncldv_list = ncldvs['genome'].to_list()
virophages = pd.read_csv("/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/ncldvs_virophages/virophage_clust_df.tsv",sep='\t')
virophage_list = virophages['votu'].to_list()

phage_list = final_votus

all_votus = phage_list + ncldv_list + virophage_list
len(all_votus)


6679

# Merging ncldvs and virophages to taxonomy table

In [33]:
ncldv_df = pd.DataFrame({
    'seq_name':ncldv_list,
    'taxonomy':'Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;'
})

viro_df = pd.DataFrame({
    'seq_name':virophage_list,
    'taxonomy':'Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;'
})

final_tax_df = pd.concat([tax_df_final, ncldv_df, viro_df])
final_tax_df = final_tax_df[['seq_name','taxonomy']]
final_tax_df['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  6034
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       410
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae                   56
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae                 23
Unclassified                                                                                         18
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;                                              17
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae                      12
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                                12
Viruses;;;;;;                                                                                        12
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudov

In [35]:
final_tax_df.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/tables/taxonomy/final_taxonomy.tsv',sep='\t', index=False)
final_tax_df

,seq_name,taxonomy
0,AP1_NODE_4947_length_11656_cov_2.939804,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
1,AP1_NODE_5071_length_11509_cov_3.290546,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
14,AP1_NODE_1422_length_23637_cov_2.675202,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
18,AP1_NODE_1411_length_23714_cov_3.915080,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
50,AP1_NODE_4021_length_13131_cov_2.998154,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...
...,...,...
7,PP2_NODE_845_length_18425_cov_2.699147,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...
8,AP1_NODE_2381_length_17703_cov_2.970642,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...
9,PP3_NODE_909_length_17486_cov_2.456363,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...
10,AP2_NODE_3393_length_15810_cov_1.154052,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...


# Merging the taxonomy to the read mapping results

In [36]:
import numpy as np

#FIltering the viral read mappings
final_viral_set = final_tax_df['seq_name'].tolist()
final_viral_set = set(final_viral_set)

#Load viral TMM normalized dataset
final_viral_map_df = pd.read_csv(
    "/MP_Data/Dados_David/viral_analysis/post_processing/viral_map_data/sectet_contigs_TMM_norm.tsv",
    sep="\t", index_col = 0
)
final_viral_map_df  = final_viral_map_df[final_viral_map_df.index.isin(final_viral_set)]
final_viral_map_df = final_viral_map_df.apply(lambda x: np.log(x+1))
final_viral_map_df.head(10)

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
AP1_NODE_1015_length_28855_cov_2.442704,3.179489,3.041338,2.529309,3.228522,2.674146,3.059092,0.890472,0.621668,0.833939,0.899279,...,0.152660,0.178573,0.120124,0.163697,0.000000,0.106693,0.024484,0.044325,0.000000,0.000000
AP1_NODE_1019_length_28805_cov_4.940442,3.361369,2.665530,2.025762,3.900340,3.263206,3.879794,2.724442,2.949828,2.487286,1.992524,...,0.000000,0.027547,0.000000,0.000000,0.000000,0.000000,0.000000,0.044325,0.049041,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,2.649467,2.500590,2.076936,2.353304,1.988366,2.329458,0.878611,1.121312,0.979303,1.068243,...,0.000000,0.000000,0.061865,0.000000,0.000000,0.000000,0.000000,0.044325,0.024821,0.000000
AP1_NODE_1087_length_27770_cov_1.242123,2.143394,2.313315,2.351496,2.103823,2.186026,2.573883,0.378169,0.338101,0.153802,0.604018,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.022408,0.000000,0.044110
AP1_NODE_10_length_320914_cov_6.491204,6.291373,5.819241,6.058626,6.896940,6.388968,7.104859,3.199745,3.262466,3.210961,3.599120,...,0.104317,0.105908,0.031411,0.000000,0.173942,0.054769,0.094531,0.147236,0.203909,0.146558
AP1_NODE_1114_length_27291_cov_2.165366,3.042769,2.697594,2.331070,3.471285,2.904739,3.177296,1.053323,1.092820,1.059276,1.135054,...,0.027117,0.054356,0.000000,0.000000,0.000000,0.106693,0.000000,0.000000,0.000000,0.000000
AP1_NODE_1116_length_27271_cov_1.756410,2.832803,2.592317,1.990130,3.315364,2.551210,2.863645,0.737916,0.681565,0.732700,0.899279,...,0.000000,0.000000,0.000000,0.057595,0.031174,0.027759,0.000000,0.022408,0.000000,0.000000
AP1_NODE_1121_length_27173_cov_1.605746,2.738088,2.652778,2.253321,3.088222,2.909174,3.148419,1.193331,1.023002,1.115319,1.197679,...,0.053518,0.080464,0.227354,0.236484,0.090748,0.156053,0.351614,0.127483,0.301471,0.106829
AP1_NODE_114_length_107093_cov_3.309715,4.534081,4.266711,3.782255,4.715905,4.452817,4.596984,2.407973,2.696062,2.558413,2.829418,...,0.000000,0.154933,0.175176,0.085195,0.173942,0.000000,0.138644,0.000000,0.072689,0.065457
AP1_NODE_1231_length_25728_cov_1.858287,2.341907,2.735436,2.128781,3.590020,2.758205,3.141067,0.925233,1.411795,1.210557,1.567654,...,0.000000,0.000000,0.031411,0.000000,0.000000,0.000000,0.000000,0.022408,0.024821,0.000000


In [39]:
final_viral_map_df = final_viral_map_df.reset_index()
final_viral_map_df = final_viral_map_df.rename(columns={'index':'contig_id'})
final_viral_map_plus_tax = pd.merge(final_tax_df, final_viral_map_df, left_on='seq_name', right_on='contig_id', how='left')
final_viral_map_plus_tax

,seq_name,taxonomy,contig_id,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
0,AP1_NODE_4947_length_11656_cov_2.939804,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,AP1_NODE_4947_length_11656_cov_2.939804,2.235626,2.005197,1.982849,2.529612,2.247908,2.549741,0.902194,...,0.000000,0.080464,0.000000,0.000000,0.000000,0.000000,0.048382,0.000000,0.000000,0.000000
1,AP1_NODE_5071_length_11509_cov_3.290546,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,AP1_NODE_5071_length_11509_cov_3.290546,2.787720,2.527964,2.336216,2.742501,2.221855,2.626676,0.936557,...,0.027117,0.000000,0.000000,0.000000,0.061405,0.000000,0.000000,0.044325,0.000000,0.000000
2,AP1_NODE_1422_length_23637_cov_2.675202,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,AP1_NODE_1422_length_23637_cov_2.675202,3.049890,2.725257,2.412763,3.410505,2.952473,3.176109,0.765014,...,0.523071,0.309754,0.276944,0.236484,0.322065,0.203091,0.419041,0.127483,0.000000,0.106829
3,AP1_NODE_1411_length_23714_cov_3.915080,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,AP1_NODE_1411_length_23714_cov_3.915080,3.021095,2.346235,2.201833,3.780947,2.632232,3.393977,1.530270,...,0.053518,0.000000,0.031411,0.029212,0.000000,0.000000,0.024484,0.000000,0.000000,0.000000
4,AP1_NODE_4021_length_13131_cov_2.998154,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,AP1_NODE_4021_length_13131_cov_2.998154,2.812767,2.463623,2.011661,2.683924,2.045944,2.421955,0.524382,...,0.152660,0.105908,0.175176,0.057595,0.061405,0.106693,0.138644,0.000000,0.049041,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6674,PP2_NODE_845_length_18425_cov_2.699147,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,PP2_NODE_845_length_18425_cov_2.699147,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.526260,1.508331,1.936600,1.721813,1.626241,1.293608,1.457805,0.819346,0.617607,0.754921
6675,AP1_NODE_2381_length_17703_cov_2.970642,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,AP1_NODE_2381_length_17703_cov_2.970642,2.764393,2.029426,1.621485,3.653849,2.024739,2.492603,0.991328,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6676,PP3_NODE_909_length_17486_cov_2.456363,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,PP3_NODE_909_length_17486_cov_2.456363,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.675809,1.657344,1.908605,1.778460,1.626241,1.246178,1.288488,0.886921,1.050738,0.700470
6677,AP2_NODE_3393_length_15810_cov_1.154052,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,AP2_NODE_3393_length_15810_cov_1.154052,1.603454,1.568467,1.243638,2.135252,1.882990,2.196384,0.737916,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.024821,0.000000


In [40]:
#final_viral_map_plus_tax = final_viral_map_plus_tax.drop('seq_name',axis=1)
final_viral_map_plus_tax = final_viral_map_plus_tax.drop('contig_id', axis=1)

In [41]:
final_tax_by_replicate = final_viral_map_plus_tax
final_tax_by_replicate = final_tax_by_replicate.drop('seq_name', axis=1) 
final_tax_by_replicate = final_tax_by_replicate.groupby('taxonomy').sum()
final_tax_by_replicate.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/votus_definition/taxonomy/viral_classification_by_replicate.tsv',sep='\t')
final_tax_by_replicate

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
taxonomy,,,,,,,,,,,,,,,,,,,,,
Unclassified,0.564833,0.523537,0.444716,0.619687,0.393291,0.413414,12.413578,13.890547,12.403361,13.686716,...,3.101069,4.044456,4.490914,4.388289,3.733146,3.517518,3.752744,3.464544,3.098532,3.199800
Viruses;;;;;;,4.043929,3.431772,2.711134,5.472164,4.656456,4.197454,7.083897,8.325587,7.327410,7.815846,...,2.226792,2.103543,2.979255,2.681996,2.142892,1.649182,1.752466,1.446838,0.791254,0.956932
Viruses;;;;;;Thaspiviridae,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.028329,0.079855,0.024824,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.048382,0.000000,0.000000,0.000000
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;,1766.793866,1536.103140,1351.016641,1995.760652,1630.361924,1821.520520,2536.587817,2694.591972,2552.456191,2550.621612,...,1014.254955,1024.069560,1096.020401,1024.251620,1063.152718,858.951469,1012.179908,1076.308696,966.233339,933.999049
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Ackermannviridae,0.000000,0.000000,0.051719,0.000000,0.000000,0.000000,1.891638,2.348553,1.873999,1.752127,...,0.000000,0.000000,0.000000,0.000000,0.061405,0.000000,0.048382,0.044325,0.000000,0.000000
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae,7.441756,6.439645,7.631868,6.962920,5.891574,6.940678,4.146546,4.925412,3.709617,5.441430,...,0.027117,0.081903,0.156097,0.225016,0.422725,0.163595,0.169072,0.155874,0.378275,0.302161
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Casjensviridae,0.000000,0.050736,0.000000,0.000000,0.000000,0.135798,1.737351,1.866834,2.154259,1.616702,...,0.053518,0.000000,0.000000,0.000000,0.061405,0.054769,0.000000,0.022408,0.049041,0.044110
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae,0.587559,0.384177,0.501681,0.440459,0.222617,0.468741,18.117019,18.955337,18.855638,17.097096,...,0.229500,0.251786,0.120124,0.143619,0.208378,0.234617,0.245912,0.618064,0.429830,0.288468
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Fredfastierviridae,0.036770,0.025690,0.000000,0.000000,0.000000,0.028046,4.475369,4.421337,4.175939,3.930714,...,0.027117,0.081903,0.031411,0.248892,0.000000,0.027759,0.024484,4.662387,4.809883,4.841379


In [ ]:
final_tax_by_sample = pd.DataFrame({
    'Taxonomy': final_viral_map_plus_tax['taxonomy'],

    # AP group
    'AP1': final_viral_map_plus_tax.iloc[:, 2:5].mean(axis=1),   
    'AP2': final_viral_map_plus_tax.iloc[:, 5:8].mean(axis=1),   
    'AP3': final_viral_map_plus_tax.iloc[:, 8:11].mean(axis=1),  
    'AP4': final_viral_map_plus_tax.iloc[:, 11:14].mean(axis=1), 
    'AP5': final_viral_map_plus_tax.iloc[:, 14:17].mean(axis=1), 

    # BP group
    'BP1': final_viral_map_plus_tax.iloc[:, 17:20].mean(axis=1),                  
    'BP2': final_viral_map_plus_tax.iloc[:, 20:23].mean(axis=1), 
    'BP3': final_viral_map_plus_tax.iloc[:, 23:26].mean(axis=1),
    'BP4': final_viral_map_plus_tax.iloc[:, 26:29].mean(axis=1),
    'BP5': final_viral_map_plus_tax.iloc[:, 29:32].mean(axis=1),

    # MP group
    'MP1': final_viral_map_plus_tax.iloc[:, 32:35].mean(axis=1),
    'MP2': final_viral_map_plus_tax.iloc[:, 35:38].mean(axis=1),
    'MP3': final_viral_map_plus_tax.iloc[:, 38:41].mean(axis=1),
    'MP4': final_viral_map_plus_tax.iloc[:, 41:44].mean(axis=1),
    'MP5': final_viral_map_plus_tax.iloc[:, 44:47].mean(axis=1),

    # PP group
    'PP1': final_viral_map_plus_tax.iloc[:, 47:50].mean(axis=1),
    'PP2': final_viral_map_plus_tax.iloc[:, 50:53].mean(axis=1),
    'PP3': final_viral_map_plus_tax.iloc[:, 53:56].mean(axis=1),
    'PP4': final_viral_map_plus_tax.iloc[:, 56:59].mean(axis=1),
    'PP5': final_viral_map_plus_tax.iloc[:, 59:62].mean(axis=1)
})
#final_tax_by_sample.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/viral_family_classification_per_sample.tsv',sep='\t',index=False)
final_tax_by_sample

,Taxonomy,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,...,MP1,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5
0,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,2.074557,2.442420,1.017369,0.988326,0.471357,0.050125,0.020784,0.023858,0.012001,...,0.080960,0.316627,0.197872,0.400782,0.480165,0.055172,0.010565,0.026821,0.016127,0.000000
1,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,2.550633,2.530344,0.770016,0.752086,2.076949,0.000000,0.000000,0.000000,0.000000,...,0.027259,0.292215,0.321865,0.358718,0.495471,0.033472,0.009039,0.000000,0.020468,0.014775
2,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,2.729303,3.179696,0.842707,0.739767,0.375010,0.000000,0.078877,0.042663,0.019914,...,0.229750,0.922228,0.755282,0.907405,1.327637,0.304974,0.504368,0.274394,0.314732,0.078104
3,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,2.523055,3.269052,1.403930,1.632321,0.442227,0.000000,0.000000,0.008051,0.000000,...,0.006545,0.033486,0.019504,0.138338,0.081622,0.025414,0.036495,0.020208,0.008161,0.000000
4,Viruses;Duplodnaviria;Heunggongvirae;Urovirico...,2.429350,2.383941,0.546372,0.471743,1.630984,0.000000,0.000000,0.000000,0.000000,...,0.123599,0.282356,0.259242,0.313867,0.425169,0.204388,0.108014,0.112893,0.102247,0.016347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6674,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.730758,1.796796,1.722248,1.459218,0.730625
6675,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,2.138434,2.723730,1.155856,1.248649,0.404822,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.074218,0.025299,0.000000,0.000000,0.000000,0.000000,0.000000
6676,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.014296,0.007178,0.000000,0.000000,0.000000,1.747171,1.828833,1.781470,1.386969,0.879376
6677,Viruses;Varidnaviria;Bamfordvirae;Preplasmivir...,1.471853,2.071542,0.664593,0.599442,0.205934,0.000000,0.000000,0.022014,0.000000,...,0.012964,0.044708,0.006726,0.014782,0.030747,0.008688,0.018655,0.000000,0.000000,0.008274


# Organizing the top 10 viral families all cities

In [ ]:
allcities_tax_abund = final_tax_by_sample
allcities_tax_abund = allcities_tax_abund.groupby('Taxonomy').sum()

allcities_tax_abund['total'] = allcities_tax_abund.sum(axis=1)
allcities_tax_abund = allcities_tax_abund.sort_values(by='total',ascending=False)
#allcities_tax_abund.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/allcities_viral_rel_abund.tsv',sep='\t')
allcities_tax_abund

,AP1,AP2,AP3,AP4,AP5,BP1,BP2,BP3,BP4,BP5,...,MP2,MP3,MP4,MP5,PP1,PP2,PP3,PP4,PP5,total
Taxonomy,,,,,,,,,,,,,,,,,,,,,
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;,1551.304549,1815.881032,2594.545327,2299.378183,3496.211277,1239.598347,1441.674142,1188.130809,1059.606104,672.413624,...,4787.098494,4215.471775,5156.254807,6311.420229,896.047193,1115.195530,1048.113860,978.094698,992.180361,48245.019371
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae,278.777082,299.034275,93.880975,83.565361,490.515612,10.575797,8.959138,6.140066,3.198169,5.739749,...,496.899572,463.448417,606.180984,742.387005,42.314412,29.255683,28.424264,20.302708,12.153365,4034.793151
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae,7.171089,6.598391,4.260525,4.550403,38.228959,0.998618,2.479525,1.309914,0.772963,0.568753,...,73.996176,67.358633,79.943112,97.403148,0.195147,0.055808,0.154338,0.251798,0.278770,427.170449
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;,12.672707,15.103760,4.403671,4.177581,11.077448,1.173112,1.303794,1.108243,0.520714,0.495232,...,45.186383,43.069038,45.866486,50.254296,6.146655,5.678718,5.927631,4.603694,3.156477,294.062362
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae,24.345684,28.202622,12.183067,12.102152,23.552302,0.465378,0.742346,0.981915,1.531068,1.524077,...,14.565554,11.976173,14.564428,18.571474,8.916772,10.435879,9.862879,8.093172,6.086183,215.618470
Unclassified,0.511029,0.475464,12.902495,11.576932,9.493333,3.882799,11.141540,8.398094,4.861399,3.505335,...,3.061433,3.194206,4.657121,4.026034,4.029458,3.772675,4.307886,3.667803,3.254292,105.197050
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae,0.491139,0.377272,18.642665,15.606754,19.155801,0.633191,1.198906,0.973617,0.857077,0.908767,...,6.588381,5.187332,7.555269,9.039773,0.290317,0.181165,0.171843,0.229636,0.445454,94.902962
Viruses;;;;;;,3.395612,4.775358,7.578965,6.754242,3.475047,0.389247,0.264549,0.220201,0.140638,0.171237,...,9.475842,8.641740,11.300124,11.229156,2.917404,2.544396,2.588264,1.848180,1.065008,83.910863
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;,5.571212,7.132951,4.930973,4.401887,7.972505,0.414055,0.614922,0.471354,0.211818,0.101279,...,8.210389,6.483028,6.316177,9.346686,3.834041,3.812181,3.729716,2.932361,1.774396,81.779396


# Organizing the top 10 viral families per city

## Altamira

In [ ]:
altamira_tax_abund = final_tax_by_sample.iloc[:, :6]
altamira_tax_abund = altamira_tax_abund.groupby('Taxonomy').sum()

altamira_tax_abund['total'] = altamira_tax_abund.sum(axis=1)
altamira_tax_abund = altamira_tax_abund.sort_values(by='total',ascending=False)
#altamira_tax_abund = altamira_tax_abund.reset_index()
#altamira_tax_abund = altamira_tax_abund.drop('Caudoviricetes')
#altamira_tax_abund = altamira_tax_abund.drop('Unclassified')
altamira_tax_abund = altamira_tax_abund.drop('total',axis=1)
altamira_tax_abund.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/altamira_viral_rel_abund.tsv',sep='\t')
total_alt = altamira_tax_abund.sum()
perc_alt = (altamira_tax_abund / total_alt) * 100
perc_alt.head(10).T.sum() / 5


Taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                             86.852299
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                  9.382277
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae           0.812167
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae             0.387792
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae                0.359945
Viruses;Monodnaviria;Sangervirae;Phixviricota;Malgrandaviricetes;Petitvirales;Microviridae     0.336246
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;                                        0.389993
Unclassified                                                                                   0.240898
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                          0.232975
Viruses;;;;;;                                          

In [ ]:
print(f"Sequências não classificadas em altamira (porcentagem): {86.852299 + 0.240898 + 0.204601}")

Sequências não classificadas em altamira (porcentagem): 87.297798


In [ ]:
alt_final_tax_df = final_tax_df[final_tax_df['seq_name'].str.startswith('AP')]
alt_final_tax_df['taxonomy'].value_counts()
#print(f"Sequências não classificadas em altamira (porcentagem): {87.216841 + 0.255728 + 0.192651}")

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  1728
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       167
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae                 14
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae                   12
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae                      10
Unclassified                                                                                          9
Viruses;Monodnaviria;Sangervirae;Phixviricota;Malgrandaviricetes;Petitvirales;Microviridae            6
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                                 6
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;;;                                                5
Viruses;Varidnaviria;Bamfordvirae;;;;                  

### Bragança

In [ ]:
braganca_tax_abund = final_tax_by_sample.iloc[:, :12]
braganca_tax_abund = braganca_tax_abund.groupby('Taxonomy').sum()
braganca_tax_abund = braganca_tax_abund.iloc[:, 5:10]

braganca_tax_abund['total'] = braganca_tax_abund.sum(axis=1)
braganca_tax_abund = braganca_tax_abund.sort_values(by='total',ascending=False)
#braganca_tax_abund = braganca_tax_abund.reset_index()
#braganca_tax_abund = braganca_tax_abund.drop('Caudoviricetes')
#braganca_tax_abund = braganca_tax_abund.drop('Unclassified')
braganca_tax_abund = braganca_tax_abund.drop('total',axis=1)
braganca_tax_abund.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/braganca_viral_rel_abund.tsv',sep='\t')

total_bra = braganca_tax_abund.sum()
perc_bra = (braganca_tax_abund / total_bra) * 100
perc_bra.head(10).T.sum() / 5

Taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  96.803440
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       0.608842
Unclassified                                                                                        0.536622
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Fredfastierviridae                 0.470600
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Tectiliviricetes;Kalamavirales;Tectiviridae     0.168258
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Schitoviridae                      0.176129
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Casjensviridae                     0.209669
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;                       0.150729
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Straboviridae                      0.144300
Viruses;Du

In [ ]:
print(f"Sequências não classificadas em braganca (porcentagem): {96.803440 + 0.536622}")

Sequências não classificadas em braganca (porcentagem): 97.34006199999999


In [ ]:
bra_final_tax_df = final_tax_df[final_tax_df['seq_name'].str.startswith('BP')]
bra_final_tax_df['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  469
Unclassified                                                                                         4
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Fredfastierviridae                  3
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                        2
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Tectiliviricetes;Kalamavirales;Tectiviridae      2
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Straboviridae                       2
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Casjensviridae                      2
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Herelleviridae                      1
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;                        1
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;

### Marabá

In [ ]:
maraba_tax_abund = final_tax_by_sample.iloc[:, :17]
maraba_tax_abund = maraba_tax_abund.groupby('Taxonomy').sum()
maraba_tax_abund = maraba_tax_abund.iloc[:, 10:15]

maraba_tax_abund['total'] = maraba_tax_abund.sum(axis=1)
maraba_tax_abund = maraba_tax_abund.sort_values(by='total',ascending=False)
#maraba_tax_abund = maraba_tax_abund.reset_index()
#maraba_tax_abund = maraba_tax_abund.drop('Caudoviricetes')
#maraba_tax_abund = maraba_tax_abund.drop('Unclassified')
maraba_tax_abund = maraba_tax_abund.drop('total',axis=1)
maraba_tax_abund.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/maraba_viral_rel_abund.tsv',sep='\t')

total_mar = maraba_tax_abund.sum()
perc_mar = (maraba_tax_abund / total_mar) * 100
perc_mar.head(10).T.sum() / 5

Taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                       87.782664
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae            8.858113
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae       1.221143
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;                                  0.743513
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae     0.225847
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;            0.159442
Viruses;;;;;;                                                                            0.156380
Viruses;Varidnaviria;Bamfordvirae;;;;                                                    0.131320
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Demerecviridae          0.117231
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                    0.115488
dtype: floa

In [ ]:
mar_final_tax_df = final_tax_df[final_tax_df['seq_name'].str.startswith('MP')]
mar_final_tax_df['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  3344
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       234
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Autographiviridae                   43
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;                                              12
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;Crassvirales;                         9
Viruses;;;;;;                                                                                         6
Viruses;Varidnaviria;Bamfordvirae;;;;                                                                 5
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Zobellviridae                        5
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Ackermannviridae                     5
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Tec

### Paragominas

In [ ]:
paragominas_tax_abund = final_tax_by_sample.iloc[:, :22]
paragominas_tax_abund = paragominas_tax_abund.groupby('Taxonomy').sum()
paragominas_tax_abund = paragominas_tax_abund.iloc[:, 15:20]

paragominas_tax_abund['total'] = paragominas_tax_abund.sum(axis=1)
paragominas_tax_abund = paragominas_tax_abund.sort_values(by='total',ascending=False)
#paragominas_tax_abund = paragominas_tax_abund.reset_index()
#paragominas_tax_abund = paragominas_tax_abund.drop('Caudoviricetes')
#paragominas_tax_abund = paragominas_tax_abund.drop('Unclassified')
paragominas_tax_abund = paragominas_tax_abund.drop('total',axis=1)
paragominas_tax_abund.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/paragominas_viral_rel_abund.tsv',sep='\t')


total_par = paragominas_tax_abund.sum()
perc_par = (paragominas_tax_abund / total_par) * 100
perc_par.head(10).T.sum() / 5

Taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  94.899070
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                       2.521296
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae                0.817179
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;                                             0.482494
Unclassified                                                                                        0.360516
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                               0.303553
Viruses;;;;;;                                                                                       0.207361
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;;;                                              0.137448
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Fredfastierviridae                 0.097529
Viruses;Va

In [ ]:
par_final_tax_df = final_tax_df[final_tax_df['seq_name'].str.startswith('PP')]
par_final_tax_df['taxonomy'].value_counts()

taxonomy
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;                                  493
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Kyanoviridae                        7
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Stanwilliamsviridae                 3
Viruses;;;;;;                                                                                        2
Unclassified                                                                                         2
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Maveriviricetes;;                                2
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;;;                                               1
Viruses;Duplodnaviria;Heunggongvirae;Uroviricota;Caudoviricetes;;Fredfastierviridae                  1
Viruses;Varidnaviria;Bamfordvirae;Preplasmiviricota;Tectiliviricetes;Vinavirales;Corticoviridae      1
Viruses;Varidnaviria;Bamfordvirae;Nucleocytoviricota;;;         

# Creating vOTUs datasets

In [ ]:
dataset1 = final_viral_map_df #6679 vOTUs (all qualities)
dataset1 = dataset1.set_index('contig_id')
dataset1.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/votus_dataset1_TMM_rel_abund.tsv',sep='\t')
dataset1

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
contig_id,,,,,,,,,,,,,,,,,,,,,
AP1_NODE_1015_length_28855_cov_2.442704,3.179489,3.041338,2.529309,3.228522,2.674146,3.059092,0.890472,0.621668,0.833939,0.899279,...,0.152660,0.178573,0.120124,0.163697,0.000000,0.106693,0.024484,0.044325,0.000000,0.000000
AP1_NODE_1019_length_28805_cov_4.940442,3.361369,2.665530,2.025762,3.900340,3.263206,3.879794,2.724442,2.949828,2.487286,1.992524,...,0.000000,0.027547,0.000000,0.000000,0.000000,0.000000,0.000000,0.044325,0.049041,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,2.649467,2.500590,2.076936,2.353304,1.988366,2.329458,0.878611,1.121312,0.979303,1.068243,...,0.000000,0.000000,0.061865,0.000000,0.000000,0.000000,0.000000,0.044325,0.024821,0.000000
AP1_NODE_1087_length_27770_cov_1.242123,2.143394,2.313315,2.351496,2.103823,2.186026,2.573883,0.378169,0.338101,0.153802,0.604018,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.022408,0.000000,0.044110
AP1_NODE_10_length_320914_cov_6.491204,6.291373,5.819241,6.058626,6.896940,6.388968,7.104859,3.199745,3.262466,3.210961,3.599120,...,0.104317,0.105908,0.031411,0.000000,0.173942,0.054769,0.094531,0.147236,0.203909,0.146558
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PP5_NODE_924_length_15762_cov_1.839591,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.206855,0.230008,0.845903,0.183221,...,0.000000,0.105908,0.120124,0.138208,0.200204,0.106693,0.048382,2.751216,2.546451,2.394855
PP5_NODE_94_length_56131_cov_2.043533,0.000000,0.025690,0.051719,0.000000,0.000000,0.000000,0.206855,0.358385,0.266058,0.140466,...,3.181796,3.117503,3.448128,3.025000,3.564572,2.665165,3.314944,2.663564,2.382003,2.260838
PP5_NODE_953_length_15547_cov_2.636252,0.000000,0.025690,0.051719,0.063380,0.052695,0.000000,0.055861,0.108798,0.053939,0.000000,...,2.370551,2.530758,2.847338,2.330030,2.596186,2.033782,2.294846,1.804357,1.474231,1.322013


### Filtering by checkv quality

In [ ]:
ncldv_qual = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/ncldv_virophage_plv_check/NCLDVs_phylogeny/checkv_results/quality_summary.tsv',sep='\t')
virophage_qual = pd.read_csv('/MP_Data/Dados_David/viral_analysis/post_processing/ncldv_virophage_plv_check/virophage_phylogeny/checkv_results/quality_summary.tsv',sep='\t')

virophage_qual = virophage_qual[virophage_qual['contig_id'].isin(virophage_list)]

all_qualities = pd.concat([quality_df_final, ncldv_qual, virophage_qual])
all_qualities = all_qualities[all_qualities['checkv_quality'].isin(['High-quality','Complete','Medium-quality'])]
votus_dataset2 = all_qualities['contig_id'].tolist()

dataset2  = final_viral_map_df[final_viral_map_df['contig_id'].isin(votus_dataset2)]
dataset2 = dataset2.set_index('contig_id')
dataset2.to_csv('/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/01_process_votus/tables/votus_dataset2_TMM_rel_abund.tsv',sep='\t')
dataset2

,AP1_A,AP1_B,AP1_C,AP2_A,AP2_B,AP2_C,AP3_A,AP3_B,AP3_C,AP4_A,...,PP2_C,PP3_A,PP3_B,PP3_C,PP4_A,PP4_B,PP4_C,PP5_A,PP5_B,PP5_C
contig_id,,,,,,,,,,,,,,,,,,,,,
AP1_NODE_1019_length_28805_cov_4.940442,3.361369,2.665530,2.025762,3.900340,3.263206,3.879794,2.724442,2.949828,2.487286,1.992524,...,0.000000,0.027547,0.000000,0.000000,0.000000,0.000000,0.000000,0.044325,0.049041,0.000000
AP1_NODE_1026_length_28694_cov_1.324710,2.649467,2.500590,2.076936,2.353304,1.988366,2.329458,0.878611,1.121312,0.979303,1.068243,...,0.000000,0.000000,0.061865,0.000000,0.000000,0.000000,0.000000,0.044325,0.024821,0.000000
AP1_NODE_10_length_320914_cov_6.491204,6.291373,5.819241,6.058626,6.896940,6.388968,7.104859,3.199745,3.262466,3.210961,3.599120,...,0.104317,0.105908,0.031411,0.000000,0.173942,0.054769,0.094531,0.147236,0.203909,0.146558
AP1_NODE_114_length_107093_cov_3.309715,4.534081,4.266711,3.782255,4.715905,4.452817,4.596984,2.407973,2.696062,2.558413,2.829418,...,0.000000,0.154933,0.175176,0.085195,0.173942,0.000000,0.138644,0.000000,0.072689,0.065457
AP1_NODE_1255_length_25433_cov_16.962025,4.637521,4.445716,4.312785,5.167857,4.646419,4.805851,1.425471,1.511678,1.405070,1.415062,...,0.027117,0.027547,0.061865,0.000000,0.000000,0.054769,0.000000,0.065772,0.000000,0.022298
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PP5_NODE_68_length_66729_cov_2.245543,0.036770,0.000000,0.051719,0.000000,0.000000,0.000000,0.082664,0.108798,0.027333,0.049047,...,0.000000,0.000000,0.061865,0.112053,0.000000,0.054769,0.000000,4.311572,3.768663,3.827825
PP5_NODE_69_length_66578_cov_17.002333,0.139603,0.189115,0.076609,0.206209,0.052695,0.157565,0.636838,0.804500,0.620044,0.631126,...,5.054460,5.117253,5.253212,5.167348,5.104352,4.471627,4.689453,4.773550,4.635317,3.932973
PP5_NODE_72_length_65542_cov_1.540702,0.139603,0.050736,0.000000,0.000000,0.000000,0.000000,1.043254,1.202217,1.251019,0.768352,...,1.154223,1.190967,1.240406,1.315645,0.996906,0.854886,1.015156,3.894510,3.649922,3.277250


# Counting number of UVIGs

In [ ]:
quality_df_final = pd.concat([quality_df_final, ncldv_qual, virophage_qual])
quality_df_final.shape

(6783, 15)

In [ ]:
quality_df_final.describe()

,contig_length,proviral_length,gene_count,viral_genes,host_genes,completeness,contamination,kmer_freq,length
count,6648.000000,237.000000,6648.000000,6648.000000,6648.000000,6590.000000,6648.000000,6648.000000,6608.000000
mean,25415.010379,25167.286920,34.416817,8.725933,0.865674,31.327502,1.074069,1.003362,25621.716858
std,27102.033969,32952.037422,36.768227,10.229580,2.663943,28.218756,6.713315,0.031808,30027.308144
min,4537.000000,3036.000000,1.000000,0.000000,0.000000,0.590000,0.000000,1.000000,4537.000000
25%,12387.500000,9701.000000,16.000000,3.000000,0.000000,9.280000,0.000000,1.000000,12384.750000
50%,16855.000000,15293.000000,24.000000,6.000000,0.000000,21.835000,0.000000,1.000000,16842.500000
75%,28974.500000,27656.000000,40.000000,11.000000,1.000000,42.315000,0.000000,1.000000,28931.750000
max,574697.000000,360459.000000,957.000000,151.000000,61.000000,100.000000,86.180000,2.010000,797112.000000


In [ ]:
all_qualities = pd.concat([quality_df_final, ncldv_qual, virophage_qual])
all_qualities = all_qualities[all_qualities['checkv_quality'].isin(['High-quality','Complete','Medium-quality'])]
all_qualities.describe()

,contig_length,proviral_length,gene_count,viral_genes,host_genes,completeness,contamination,kmer_freq,length
count,1348.000000,34.000000,1348.000000,1348.000000,1348.000000,1348.000000,1348.000000,1348.000000,1308.000000
mean,48577.362760,56553.000000,66.618694,16.978487,1.281899,80.159800,0.502226,1.008375,49211.811927
std,46366.195992,63479.104055,61.312361,16.855215,4.390115,18.282251,4.235447,0.064055,51524.485945
min,4537.000000,6847.000000,7.000000,0.000000,0.000000,50.020000,0.000000,1.000000,4537.000000
25%,27081.750000,31232.750000,38.000000,8.000000,0.000000,61.757500,0.000000,1.000000,27752.500000
50%,38136.500000,39868.500000,54.000000,13.000000,0.000000,82.270000,0.000000,1.000000,38311.000000
75%,51562.750000,47398.500000,74.000000,20.000000,1.000000,100.000000,0.000000,1.000000,51745.000000
max,574697.000000,360459.000000,957.000000,151.000000,61.000000,100.000000,67.830000,2.000000,797112.000000


# Counting the number of UViGs

In [43]:
clust_df = pd.read_csv('/MP_Data/Dados_David/viral_analysis/results/clustering/all_viral_contigs_clusters.tsv',sep='\t')
clust_df_filtered = clust_df[clust_df['cluster '].isin(all_votus)]
clust_df_filtered['elements_number'] = clust_df_filtered['components'].str.count(',') + 1
clust_df_filtered['elements_number'].sum()
 

/MP_Data/database/rgi/wildcard/custom_tmp/ipykernel_866258/1715463718.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clust_df_filtered['elements_number'] = clust_df_filtered['components'].str.count(',') + 1


np.int64(17277)

## Extracting the viral sequences (UViGs/scaffolds)

In [45]:
scaffolds = []
for cluster in clust_df_filtered['components'].str.split(','):
    for scaffold in cluster:
        scaffolds.append(scaffold)


scaffold_records=[]
for record in SeqIO.parse('/MP_Data/Dados_David/viral_analysis/results/clustering/concatenated_merged_viral_contigs.fasta','fasta'):
    if record.id in scaffolds:
        scaffold_records.append(record)
SeqIO.write(scaffold_records,'/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/sequences/final_scaffolds.fasta', 'fasta')

17277

## Extracting the votus

In [46]:
from Bio import SeqIO
records=[]
for record in SeqIO.parse('/MP_Data/Dados_David/viral_analysis/results/miuvigs.fasta','fasta'):
    if record.id in all_votus:
        records.append(record)
SeqIO.write(records,'/MP_Data/Dados_David/viral_analysis/post_processing/article_scripts/data_analysis/01_process_votus/sequences/final_votus.fasta', 'fasta')

6679